In [ ]:
import os
import SimpleITK as sitk
import numpy as np
import glob

path = r'/home/chaimeleon/persistent-home/classification_model_originalimage/totalsegment_work/resampled/train_val'
female_paths = glob.glob(os.path.join(path, 'female_train_val', '*', 'resampled_image.nii.gz'))
male_paths = glob.glob(os.path.join(path, 'male_train_val', '*', 'resampled_image.nii.gz'))
image_paths =female_paths + male_paths
print(image_paths)


In [ ]:
def derive_patch_path(original_path, patch_dir):
    """
    Derive the patch path from the original image path.
    :param original_path: Path to the original image.
    :return: Path to the corresponding patch image.
    """
    base_name = original_path.split('/')[-2]
    # patch_dir = "/home/resampled/train_val_patches/all_patches"
    patch_path = os.path.join(patch_dir, f"{base_name}.nii.gz")
    return patch_path

In [ ]:
def center_crop_around_rectum_patch(original_img, rectum_patch_img, crop_size=(256,256,48)):
   
    # Basic info about the rectum patch
    patch_size = rectum_patch_img.GetSize()              # (width, height, depth) in index space
    patch_spacing = rectum_patch_img.GetSpacing()        # mm per voxel
    patch_origin = rectum_patch_img.GetOrigin()          # physical origin
    patch_direction = rectum_patch_img.GetDirection()    # direction cosines
    
    # -- Step 2: Compute the center of the rectum patch
    
    # (A) If the patch is already a binary mask or label, you could use a LabelShapeStatistics approach.
    #     For example:
    #
    #     label_stats = sitk.LabelShapeStatisticsImageFilter()
    #     label_stats.Execute(rectum_patch_img)
    #     # Assuming '1' is your rectum label value:
    #     patch_center_physical = label_stats.GetCenterOfMass(1)
    #
    # (B) Otherwise, if the patch is itself an image with intensity values, 
    #     you may simply take the geometric center (center of the bounding box):
    patch_center_index = [ s // 2 for s in patch_size ]  # (cx_idx, cy_idx, cz_idx)
    # Transform the center from index space -> physical space
    patch_center_physical = rectum_patch_img.TransformContinuousIndexToPhysicalPoint(patch_center_index)
    
    # -- Step 3: Convert that center to the *original image's* index space
    #
    # Because we assume rectum_patch_img is in the same physical space (same origin, direction, spacing),
    # we can directly map that physical point to the index of original_img:
    center_in_original_index = original_img.TransformPhysicalPointToIndex(patch_center_physical)
    
    # -- Step 4: Perform center crop of size = crop_size = (224,224,48) in the original image
    
    # Half sizes
    half_size = [c // 2 for c in crop_size]
    
    # Compute starting index in the original image, ensuring we don't go out of bounds
    start_index = [
        max(center_in_original_index[0] - half_size[0], 0),
        max(center_in_original_index[1] - half_size[1], 0),
        max(center_in_original_index[2] - half_size[2], 0)
    ]
    
    # Adjust actual ROI size if near boundary
    # (ROI must be fully inside the image)
    max_index = list(original_img.GetSize())  # e.g. [width, height, depth]
    
    roi_size = [
        min(crop_size[0], max_index[0] - start_index[0]),
        min(crop_size[1], max_index[1] - start_index[1]),
        min(crop_size[2], max_index[2] - start_index[2])
    ]
    
    # ROI filter
    roi_filter = sitk.RegionOfInterestImageFilter()
    roi_filter.SetIndex(start_index)
    roi_filter.SetSize(roi_size)
    
    cropped_img = roi_filter.Execute(original_img)

    return cropped_img


# --------------------------
# Example usage for a folder:
# --------------------------
rectum_patch_folder = r"/home/chaimeleon/persistent-home/classification_model_originalimage/totalsegment_work/resampled/train_val_patches/all_patches"

output_folder = r'/home/chaimeleon/persistent-home/classification_model_originalimage/totalsegment_work/resampled/centrecrop_correct3/train_val'

for img_path in image_paths:
    img = sitk.ReadImage(img_path)
    patch_img_path = derive_patch_path(img_path, rectum_patch_folder)
    patch_img = sitk.ReadImage(patch_img_path)

    # Perform center cropping on the original image
    cropped_image = center_crop_around_rectum_patch(img, patch_img)

    # Save the cropped image
    file_name = img_path.split('/')[-2] + '.nii.gz' #patient number
    output_path = os.path.join(output_folder, file_name)

    sitk.WriteImage(cropped_image, output_path)
    print(f"Cropped image saved to {output_path}")

print('done')


